[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module2/01-iterators-generators.ipynb)

# Iterators & Generators
**Module 2 — Intermediate Python | Estimated time: 25 minutes**

## Learning Objectives
- Understand the **iterator protocol** (`__iter__` / `__next__`) and implement a custom iterator class
- Use the **`yield` keyword** to convert a function into a generator
- Compare **generator expressions** vs list comprehensions for memory efficiency
- Apply `yield from` and build **infinite generators**
- Work with **`itertools`** — `chain`, `islice`, `product`, `combinations`, `permutations`, `groupby`

In [ ]:
import sys
import itertools
from typing import Iterator, Generator

print('Python version:', sys.version)
print('Setup complete.')

## 1. The Iterator Protocol

An **iterable** is any object that can return an iterator (it implements `__iter__`).  
An **iterator** is an object that remembers its position and returns the next value via `__next__`.  
When no more values exist, `__next__` must raise `StopIteration`.

Every `for` loop silently calls `iter(obj)` and then repeatedly calls `next()` until `StopIteration` is caught.

In [ ]:
class CountUp:
    """Custom iterator that counts from `start` up to (but not including) `stop`."""

    def __init__(self, start: int, stop: int, step: int = 1):
        self.current = start
        self.stop = stop
        self.step = step

    # __iter__ returns the iterator object itself
    def __iter__(self) -> 'CountUp':
        return self

    # __next__ produces the next value or signals exhaustion
    def __next__(self) -> int:
        if self.current >= self.stop:
            raise StopIteration          # signal that iteration is done
        value = self.current
        self.current += self.step
        return value


# Using a for loop (pythonic — StopIteration is caught automatically)
print('for loop:', end=' ')
for n in CountUp(0, 10, 2):
    print(n, end=' ')
print()

# Manually calling next() to see the protocol at work
it = CountUp(1, 5)
print('Manual calls:', next(it), next(it), next(it), next(it))
try:
    next(it)          # one call too many
except StopIteration:
    print('StopIteration raised — iteration complete')

# Built-in iter() + next() work on any iterable
letters = iter(['a', 'b', 'c'])
print('List iterator:', next(letters), next(letters))

## 2. The `yield` Keyword — Generator Functions

Writing `__iter__` / `__next__` by hand is verbose.  
**Generator functions** let Python build the iterator machinery automatically:
- Any function containing `yield` becomes a generator function.
- Calling it returns a **generator object** (which is both an iterable and an iterator).
- Execution is **suspended** at each `yield` and **resumed** on the next `next()` call.

In [ ]:
def count_up(start: int, stop: int, step: int = 1) -> Generator[int, None, None]:
    """Generator version of CountUp — far less boilerplate."""
    current = start
    while current < stop:
        yield current          # suspend here, hand value to caller
        current += step


gen = count_up(0, 10, 2)
print(type(gen))               # <class 'generator'>
print(list(gen))               # consume the whole generator into a list

# Generators are lazy — values are produced only when requested
def verbose_gen():
    print('  About to yield 1')
    yield 1
    print('  About to yield 2')
    yield 2
    print('  About to yield 3')
    yield 3

print('\nLazy evaluation demo:')
for v in verbose_gen():
    print(f'  Got {v}')

## 3. Generator Expressions vs List Comprehensions — Memory

A **list comprehension** builds the entire list in memory at once.  
A **generator expression** (same syntax but with parentheses) computes values on demand.  
For large data sets the memory difference is dramatic.

In [ ]:
import sys

N = 1_000_000

# List comprehension — all N integers in RAM
big_list = [x * x for x in range(N)]

# Generator expression — only the recipe is stored
big_gen = (x * x for x in range(N))

print(f'List size:      {sys.getsizeof(big_list):>12,} bytes')
print(f'Generator size: {sys.getsizeof(big_gen):>12,} bytes')
print(f'Memory ratio:   {sys.getsizeof(big_list) / sys.getsizeof(big_gen):,.0f}x larger for the list')

# Both are iterable — sum() works on either
print(f'\nSum (list):      {sum(big_list)}')
print(f'Sum (generator): {sum(x * x for x in range(N))}')   # re-create; big_gen is exhausted

## 4. `yield from` — Delegating to a Sub-Generator

`yield from iterable` is shorthand for a loop that yields each item from the sub-iterable.  
It also transparently forwards `.send()` / `.throw()` calls to the inner generator.

In [ ]:
def flatten(nested):
    """Recursively flatten an arbitrarily nested list using yield from."""
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)   # delegate to recursive call
        else:
            yield item


data = [1, [2, 3], [4, [5, 6, [7, 8]]], 9]
print('Flattened:', list(flatten(data)))


# yield from with two generators concatenated
def combined():
    yield from range(1, 4)
    yield from ['a', 'b', 'c']

print('Combined: ', list(combined()))

## 5. Infinite Generators

Generators don't have to terminate.  
Pair them with `itertools.islice` (or a `break`) to take only what you need.

In [ ]:
import itertools

# --- Custom infinite counter ---
def integers_from(start: int = 0):
    n = start
    while True:
        yield n
        n += 1

# Take first 8 even numbers
evens = (n for n in integers_from() if n % 2 == 0)
print('First 8 evens:', list(itertools.islice(evens, 8)))

# --- itertools built-ins ---
# itertools.count  — infinite arithmetic sequence
print('count(10, 3):', list(itertools.islice(itertools.count(10, 3), 6)))

# itertools.cycle  — repeat an iterable forever
colors = itertools.cycle(['red', 'green', 'blue'])
print('cycle (9):   ', [next(colors) for _ in range(9)])

# itertools.repeat — repeat a single value
print('repeat:      ', list(itertools.repeat('x', 5)))

## 6. `itertools` — Combinatoric and Slicing Tools

`itertools` is part of the standard library and provides fast, memory-efficient building blocks.

In [ ]:
from itertools import chain, islice, product, combinations, permutations, groupby

# chain — lazily concatenate multiple iterables
print('chain:', list(chain([1, 2], [3, 4], [5])))

# product — Cartesian product (nested-loop replacement)
print('product:', list(product('AB', [1, 2])))

# combinations — order doesn't matter, no repeats
print('combinations(3,2):', list(combinations([1, 2, 3, 4], 2)))

# permutations — order matters
print('permutations(3,2):', list(permutations('ABC', 2)))

# groupby — group consecutive elements that share a key
# NOTE: input must be sorted by the key first
words = ['apple', 'ant', 'bear', 'bee', 'cat', 'cobra']
words.sort(key=lambda w: w[0])
print('\ngroupby first letter:')
for letter, group in groupby(words, key=lambda w: w[0]):
    print(f'  {letter}: {list(group)}')

## 7. Practical Example — Chunked File Reader

A real-world use case: reading a large file in fixed-size chunks without loading it all into memory.

In [ ]:
import io
import itertools

def chunked(iterable, size: int):
    """Yield successive chunks of `size` from an iterable."""
    it = iter(iterable)
    while True:
        chunk = list(itertools.islice(it, size))
        if not chunk:
            return
        yield chunk


# Simulate a large dataset (in practice this would be lines from a file)
rows = list(range(1, 26))   # 25 rows

print('Processing in batches of 7:')
for batch_num, batch in enumerate(chunked(rows, 7), start=1):
    print(f'  Batch {batch_num}: {batch}')


# Generator pipeline: read → filter → transform (all lazy)
def read_numbers(n):
    yield from range(n)

def only_odds(source):
    for x in source:
        if x % 2 != 0:
            yield x

def squared(source):
    for x in source:
        yield x ** 2

pipeline = squared(only_odds(read_numbers(20)))
print('\nPipeline result:', list(pipeline))

## Practice Exercises

**Exercise 1 — Fibonacci Generator**  
Write a generator function `fibonacci()` that yields the Fibonacci sequence infinitely (0, 1, 1, 2, 3, 5, …). Use `itertools.islice` to print the first 15 numbers.

**Exercise 2 — Running Average**  
Write a generator function `running_average(numbers)` that takes an iterable of numbers and yields the cumulative average after each new value. For input `[10, 20, 30, 40]` it should yield `10.0`, `15.0`, `20.0`, `25.0`.

**Exercise 3 — itertools Pipeline**  
You have two lists of strings: `first_names = ['Alice', 'Bob', 'Carol']` and `last_names = ['Smith', 'Jones']`. Using only `itertools.product` and a generator expression (no explicit for loops), create a list of all possible full names (e.g. `'Alice Smith'`). Then use `itertools.groupby` to group them by first name.